## Converting SMAP to parquet files ##
This notebook works to convert L4 SMAP files to parquet files. 

At the end of it, L4 SMAP files from March to July of 2024 were combined into one df 

In [3]:
import pandas as pd 
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import h5py
from tqdm import tqdm

### investigating one SMAP file first

In [2]:
file_path='/Volumes/share-sat2/SAT/SMAP/L4/2024/02/SMAP_L4_SM_gph_20240229T223000_Vv7031_001.h5'
file=h5py.File(file_path,'r')

In [3]:
file

<HDF5 file "SMAP_L4_SM_gph_20240229T223000_Vv7031_001.h5" (mode r)>

In [4]:
def print_h5_structure(name, obj):
    print(name)

with h5py.File(file_path, 'r') as file:
    print("file structure:")
    file.visititems(print_h5_structure)

file structure:
EASE2_global_projection
Geophysical_Data
Geophysical_Data/baseflow_flux
Geophysical_Data/depth_to_water_table_from_surface_in_peat
Geophysical_Data/free_surface_water_on_peat_flux
Geophysical_Data/heat_flux_ground
Geophysical_Data/heat_flux_latent
Geophysical_Data/heat_flux_sensible
Geophysical_Data/height_lowatmmodlay
Geophysical_Data/land_evapotranspiration_flux
Geophysical_Data/land_fraction_saturated
Geophysical_Data/land_fraction_snow_covered
Geophysical_Data/land_fraction_unsaturated
Geophysical_Data/land_fraction_wilting
Geophysical_Data/leaf_area_index
Geophysical_Data/mwrtm_vegopacity
Geophysical_Data/net_downward_longwave_flux
Geophysical_Data/net_downward_shortwave_flux
Geophysical_Data/overland_runoff_flux
Geophysical_Data/precipitation_total_surface_flux
Geophysical_Data/radiation_longwave_absorbed_flux
Geophysical_Data/radiation_shortwave_downward_flux
Geophysical_Data/sm_profile
Geophysical_Data/sm_profile_pctl
Geophysical_Data/sm_profile_wetness
Geophysi

In [5]:
with h5py.File(file_path, 'r') as file:
    dataset = file['/Geophysical_Data/sm_surface']
    print("shape:", dataset.shape)
    print("data type:", dataset.dtype)
    print("attributes:", dict(dataset.attrs))

shape: (1624, 3856)
data type: float32
attributes: {'DIMENSION_LABELS': array([b'y', b'x'], dtype='|S1'), 'DIMENSION_LIST': array([array([<HDF5 object reference>], dtype=object),
       array([<HDF5 object reference>], dtype=object)], dtype=object), '_FillValue': -9999.0, 'coordinates': b'/cell_lat /cell_lon', 'fmissing_value': -9999.0, 'grid_mapping': b'EASE2_global_projection', 'long_name': b'Top layer soil moisture (0-5 cm)', 'missing_value': -9999.0, 'units': b'm3 m-3', 'valid_max': 0.9, 'valid_min': 0.0}


In [6]:
with h5py.File(file_path, 'r') as file:
    dataset = file['/Geophysical_Data/sm_rootzone']
    print("shape:", dataset.shape)
    print("data type:", dataset.dtype)
    print("attributes:", dict(dataset.attrs))

shape: (1624, 3856)
data type: float32
attributes: {'DIMENSION_LABELS': array([b'y', b'x'], dtype='|S1'), 'DIMENSION_LIST': array([array([<HDF5 object reference>], dtype=object),
       array([<HDF5 object reference>], dtype=object)], dtype=object), '_FillValue': -9999.0, 'coordinates': b'/cell_lat /cell_lon', 'fmissing_value': -9999.0, 'grid_mapping': b'EASE2_global_projection', 'long_name': b'Root zone soil moisture (0-100 cm)', 'missing_value': -9999.0, 'units': b'm3 m-3', 'valid_max': 0.9, 'valid_min': 0.0}


In [7]:
with h5py.File(file_path, 'r') as file:
    dataset = file['/Geophysical_Data/surface_temp']
    print("shape:", dataset.shape)
    print("data type:", dataset.dtype)
    print("attributes:", dict(dataset.attrs))

shape: (1624, 3856)
data type: float32
attributes: {'DIMENSION_LABELS': array([b'y', b'x'], dtype='|S1'), 'DIMENSION_LIST': array([array([<HDF5 object reference>], dtype=object),
       array([<HDF5 object reference>], dtype=object)], dtype=object), '_FillValue': -9999.0, 'coordinates': b'/cell_lat /cell_lon', 'fmissing_value': -9999.0, 'grid_mapping': b'EASE2_global_projection', 'long_name': b'Mean land surface temperature (incl. snow-covered land area). Excluding areas of open water and permanent ice.', 'missing_value': -9999.0, 'units': b'K', 'valid_max': 350.0, 'valid_min': 180.0}


In [8]:
with h5py.File(file_path, 'r') as file:
    dataset = file['cell_lat']
    print("shape:", dataset.shape)
    print("data type:", dataset.dtype)
    print("attributes:", dict(dataset.attrs))

shape: (1624, 3856)
data type: float32
attributes: {'DIMENSION_LABELS': array([b'y', b'x'], dtype='|S1'), 'DIMENSION_LIST': array([array([<HDF5 object reference>], dtype=object),
       array([<HDF5 object reference>], dtype=object)], dtype=object), '_FillValue': -9999.0, 'fmissing_value': -9999.0, 'grid_mapping': b'EASE2_global_projection', 'long_name': b'The geodetic latitude of the center of each cell in the cylindrical 9 km Earth-fixed EASE-Grid 2.0. Zero latitude represents the Equator. Positive latitudes represent locations North of the Equator. Negative latitudes represent locations South of the Equator.', 'missing_value': -9999.0, 'units': b'degrees', 'valid_max': 90.0, 'valid_min': -90.0}


In [9]:
with h5py.File(file_path, 'r') as file:
    dataset = file['cell_lon']
    print("shape:", dataset.shape)
    print("data type:", dataset.dtype)
    print("attributes:", dict(dataset.attrs))

shape: (1624, 3856)
data type: float32
attributes: {'DIMENSION_LABELS': array([b'y', b'x'], dtype='|S1'), 'DIMENSION_LIST': array([array([<HDF5 object reference>], dtype=object),
       array([<HDF5 object reference>], dtype=object)], dtype=object), '_FillValue': -9999.0, 'fmissing_value': -9999.0, 'grid_mapping': b'EASE2_global_projection', 'long_name': b'The longitude of the center of each cell in the cylindrical 9 km Earth-fixed EASE-Grid 2.0. Zero longitude represents the Prime Meridian. Positive longitudes represent locations to the East of the Prime Meridian. Negative longitudes represent locations to the West of the Prime Meridian.', 'missing_value': -9999.0, 'units': b'degrees', 'valid_max': 179.999, 'valid_min': -180.0}


In [10]:
with h5py.File(file_path, 'r') as file:
    dataset = file['time']
    print("shape:", dataset.shape)
    print("data type:", dataset.dtype)
    print("attributes:", dict(dataset.attrs))

shape: (1,)
data type: float64
attributes: {'actual_range': array([7.625178e+08, 7.625178e+08]), 'delta_t': b'0000-00-00 03:00:00', 'long_name': b'Time', 'units': b'seconds since 2000-01-01 11:58:55.816'}


In [11]:
with h5py.File(file_path, 'r') as file:
    dataset = file['Geophysical_Data/leaf_area_index']
    print("shape:", dataset.shape)
    print("data type:", dataset.dtype)
    print("attributes:", dict(dataset.attrs))

shape: (1624, 3856)
data type: float32
attributes: {'DIMENSION_LABELS': array([b'y', b'x'], dtype='|S1'), 'DIMENSION_LIST': array([array([<HDF5 object reference>], dtype=object),
       array([<HDF5 object reference>], dtype=object)], dtype=object), '_FillValue': -9999.0, 'coordinates': b'/cell_lat /cell_lon', 'fmissing_value': -9999.0, 'grid_mapping': b'EASE2_global_projection', 'long_name': b'Vegetation leaf area index', 'missing_value': -9999.0, 'units': b'm2 m-2', 'valid_max': 10.0, 'valid_min': 0.0}


In [12]:
with h5py.File(file_path, 'r') as f:
    sm_surface = f['Geophysical_Data/sm_surface'][:]  #surface soil moisture
    surface_temp = f['Geophysical_Data/surface_temp'][:] #suface temperature
    LAI = f['Geophysical_Data/leaf_area_index'][:] 
    cell_lat = f['cell_lat'][:]  #latitude
    cell_lon = f['cell_lon'][:]  #longitude
    time = f['time'][:]  #time 

    print("sm_surface:", sm_surface.shape)
    print("surface_temp:", surface_temp.shape)
    print("LAI:", LAI.shape)
    print("cell_lat:", cell_lat.shape)
    print("cell_lon:", cell_lon.shape)
    print("time:", time)

sm_surface: (1624, 3856)
surface_temp: (1624, 3856)
LAI: (1624, 3856)
cell_lat: (1624, 3856)
cell_lon: (1624, 3856)
time: [7.625178e+08]


In [13]:
with h5py.File(file_path, 'r') as f:
    sm_surface = f['Geophysical_Data/sm_surface'][:]
    surface_temp = f['Geophysical_Data/surface_temp'][:]
    LAI = f['Geophysical_Data/leaf_area_index'][:]
    cell_lat = f['cell_lat'][:]
    cell_lon = f['cell_lon'][:]
    time = f['time'][:]

#.ravel() will flatten multi D arrays to 1D
df = pd.DataFrame({'latitude': cell_lat.ravel(),
                   'longitude': cell_lon.ravel(),
                   'time': np.full(cell_lat.size, time[0]),  #same time for all points because one file has one time for everything
                   'sm_surface': sm_surface.ravel(),
                   'surface_temp': surface_temp.ravel(),
                   'LAI': LAI.ravel()})

df['datetime'] = pd.to_datetime(df['time'], unit='s', origin='2000-01-01 11:58:55.816')
df.drop(columns='time', inplace=True)

In [14]:
df

,latitude,longitude,sm_surface,surface_temp,LAI,datetime
0,84.656441,-179.953323,-9999.0,-9999.0,-9999.0,2024-02-29 22:28:55
1,84.656441,-179.859970,-9999.0,-9999.0,-9999.0,2024-02-29 22:28:55
2,84.656441,-179.766602,-9999.0,-9999.0,-9999.0,2024-02-29 22:28:55
3,84.656441,-179.673248,-9999.0,-9999.0,-9999.0,2024-02-29 22:28:55
4,84.656441,-179.579880,-9999.0,-9999.0,-9999.0,2024-02-29 22:28:55
...,...,...,...,...,...,...
6262139,-84.656441,179.579880,-9999.0,-9999.0,-9999.0,2024-02-29 22:28:55
6262140,-84.656441,179.673248,-9999.0,-9999.0,-9999.0,2024-02-29 22:28:55
6262141,-84.656441,179.766602,-9999.0,-9999.0,-9999.0,2024-02-29 22:28:55
6262142,-84.656441,179.859970,-9999.0,-9999.0,-9999.0,2024-02-29 22:28:55


In [15]:
df['sm_surface'].describe()

count    6.262144e+06
mean    -7.308879e+03
std      4.434788e+03
min     -9.999000e+03
25%     -9.999000e+03
50%     -9.999000e+03
75%      4.732700e-02
max      9.016671e-01
Name: sm_surface, dtype: float64

In [16]:
## need to filter for midwest!! also should remove -9999 values 

## function to filter SMAP files from a folder and combine into a parquet file

this function works to combine all SMAP files from a folder and make it into a dataframe that can be saved as a parquet file. 

In summary, the function: 
*  takes the necessary variables from the SMAP files like surface soil moisture, rootzone soil moisture, surface temp, latitude, longitude, and time
*  filters over the midwest
*  removes missing values (-9999) by replacing them with NaN and then removing rows with Nan
*  saving output to a single parquet file 

In [5]:
def process_smap_folder(folder_path,output_parquet, lat_min, lat_max, lon_min, lon_max, 
                        time_origin='2000-01-01 11:58:55.816',missing_value=-9999):

    files = []
    for root, dirs, filenames in os.walk(folder_path):
        for f in filenames:
            if f.endswith('.h5'):
                files.append(os.path.join(root, f))
    files.sort()

    df_list = []

    for file in tqdm(files, desc="Processing files"):
        with h5py.File(file, 'r') as f:
            sm_surface = f['Geophysical_Data/sm_surface'][:]
            sm_rootzone = f['Geophysical_Data/sm_rootzone'][:]
            surface_temp = f['Geophysical_Data/surface_temp'][:]
            soil_temp_layer1 = f['Geophysical_Data/soil_temp_layer1'][:]
            LAI = f['Geophysical_Data/leaf_area_index'][:]
            cell_lat = f['cell_lat'][:]
            cell_lon = f['cell_lon'][:]
            time = f['time'][:]
            
            sm_surface =np.where(sm_surface== missing_value, np.nan, sm_surface)
            sm_rootzone=np.where(sm_rootzone==missing_value,np.nan, sm_rootzone)
            surface_temp =np.where(surface_temp== missing_value, np.nan, surface_temp)
            soil_temp_layer1 =np.where(soil_temp_layer1== missing_value, np.nan, soil_temp_layer1)
            LAI =np.where(LAI== missing_value, np.nan, LAI)

            df = pd.DataFrame({
                'latitude': cell_lat.ravel(),
                'longitude':cell_lon.ravel(),
                'sm_surface':sm_surface.ravel(),
                'sm_rootzone':sm_rootzone.ravel(),
                'surface_temp': surface_temp.ravel(),
                'soil_temp_layer1': soil_temp_layer1.ravel(),
                'LAI': LAI.ravel(),
                'time': np.full(cell_lat.size, time[0])})

            df['datetime'] = pd.to_datetime(df['time'], unit='s', origin=time_origin)
            df=df.dropna(subset=['sm_surface','sm_rootzone','surface_temp','LAI'])
            df.drop(columns='time', inplace=True)

            midwest_df =df[
                (df['latitude'] >= lat_min) &
                (df['latitude'] <= lat_max) &
                (df['longitude'] >= lon_min) &
                (df['longitude'] <= lon_max)]
            
            df_list.append(midwest_df)

    final_df = pd.concat(df_list, ignore_index=True)
    final_df.to_parquet(output_parquet)

    print(f"All SMAP files have been saved to: {output_parquet}")
    return final_df

In [6]:
#TEST 
folder_path = '/Volumes/share-sat2/SAT/SMAP/L4/2024/02'
output_parquet = '/Volumes/share-sat2/SAT/SMAP/L4/2024/02/nevada_all_SMAP_test.parquet'

lon_min, lon_max = -121, -114 
lat_min, lat_max = 35, 42

process_smap_folder(folder_path,output_parquet, lat_min, lat_max, lon_min, lon_max, 
                        time_origin='2000-01-01 11:58:55.816',missing_value=-9999)

Processing files: 100%|█████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.53s/it]

All SMAP files have been saved to: /Volumes/share-sat2/SAT/SMAP/L4/2024/02/nevada_all_SMAP_test.parquet


,latitude,longitude,sm_surface,sm_rootzone,surface_temp,soil_temp_layer1,LAI,datetime
0,41.948936,-120.949173,0.335832,0.314993,274.596100,275.466675,0.147499,2024-02-29 22:28:55
1,41.948936,-120.855812,0.315434,0.289257,274.334320,275.149689,0.154057,2024-02-29 22:28:55
2,41.948936,-120.762451,0.255968,0.217665,274.290863,275.012970,0.258017,2024-02-29 22:28:55
3,41.948936,-120.669090,0.255209,0.210780,274.463654,275.014374,0.292997,2024-02-29 22:28:55
4,41.948936,-120.575729,0.256108,0.209730,274.545258,274.652496,0.260409,2024-02-29 22:28:55
...,...,...,...,...,...,...,...,...
5835,35.034153,-114.413902,0.074427,0.110634,298.551331,292.940216,0.240138,2024-02-29 22:28:55
5836,35.034153,-114.320541,0.154119,0.244712,295.864105,290.521851,0.282281,2024-02-29 22:28:55
5837,35.034153,-114.227180,0.084441,0.162064,294.044800,289.566711,0.288873,2024-02-29 22:28:55
5838,35.034153,-114.133820,0.085660,0.164445,293.335785,288.932617,0.310394,2024-02-29 22:28:55


In [7]:
folder_path = '/Volumes/share-sat2/SAT/SMAP/L4/2024'
output_parquet = '/Volumes/share-sat2/SAT/SMAP/L4/2024/02/nevada_all_SMAP_202403-202407.parquet'
lon_min, lon_max = -121, -114 
lat_min, lat_max = 35, 42

process_smap_folder(folder_path,output_parquet, lat_min, lat_max, lon_min, lon_max, 
                        time_origin='2000-01-01 11:58:55.816',missing_value=-9999)

Processing files: 100%|█████████████████████████████████████████████████████████████████| 1225/1225 [1:20:15<00:00,  3.93s/it]


All SMAP files have been saved to: /Volumes/share-sat2/SAT/SMAP/L4/2024/02/nevada_all_SMAP_202403-202407.parquet


,latitude,longitude,sm_surface,sm_rootzone,surface_temp,soil_temp_layer1,LAI,datetime
0,41.948936,-120.949173,0.335832,0.314993,274.596100,275.466675,0.147499,2024-02-29 22:28:55
1,41.948936,-120.855812,0.315434,0.289257,274.334320,275.149689,0.154057,2024-02-29 22:28:55
2,41.948936,-120.762451,0.255968,0.217665,274.290863,275.012970,0.258017,2024-02-29 22:28:55
3,41.948936,-120.669090,0.255209,0.210780,274.463654,275.014374,0.292997,2024-02-29 22:28:55
4,41.948936,-120.575729,0.256108,0.209730,274.545258,274.652496,0.260409,2024-02-29 22:28:55
...,...,...,...,...,...,...,...,...
7153995,35.034153,-114.413902,0.022880,0.071186,322.660858,315.051605,0.191004,2024-07-31 22:28:55
7153996,35.034153,-114.320541,0.056315,0.194738,318.244049,311.383392,0.219785,2024-07-31 22:28:55
7153997,35.034153,-114.227180,0.031561,0.115976,316.065369,310.012970,0.205899,2024-07-31 22:28:55
7153998,35.034153,-114.133820,0.036369,0.122955,315.611206,309.498840,0.201426,2024-07-31 22:28:55


In [ ]:
# moved to combined 